# Aula 3, Planning

Notebook executável que acompanha a aula [03-planning.md](../../lessons/modulo-10-agentes/03-planning.md).

## O que vamos fazer aqui

Tarefas complexas exigem vários passos. Vamos construir um agente de múltiplas etapas que decompõe
um problema, executa uma conta, usa o resultado na próxima, e então responde. Python puro.

## Agente de múltiplas etapas

Cada passo encadeia o resultado do anterior. O template do passo pode usar {r} para inserir o
resultado já obtido.

In [ ]:
import ast
import operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calcular(expressao):
    def ev(no):
        if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
            return no.value
        if isinstance(no, ast.BinOp) and type(no.op) in OPS:
            return OPS[type(no.op)](ev(no.left), ev(no.right))
        raise ValueError("expressão não permitida")
    return ev(ast.parse(str(expressao), mode="eval").body)


def agente_multietapas(passos, limite=5):
    """Executa um plano passo a passo, encadeando os resultados."""
    historico = []
    resultado = None
    for i, (descricao, template) in enumerate(passos):
        if i >= limite:
            break
        expressao = template.format(r=resultado) if resultado is not None else template
        resultado = calcular(expressao)
        historico.append((descricao, expressao, resultado))
    return historico, resultado


plano = [
    ("Total de cadernos: alunos x cadernos por aluno", "28*3"),
    ("Pacotes necessários: total / cadernos por pacote", "{r}/4"),
]

historico, resposta = agente_multietapas(plano)
for descricao, expr, res in historico:
    print(f"- {descricao}: {expr} = {res}")
print("Resposta final:", resposta, "pacotes")

O agente executa 28*3 = 84 cadernos e usa o resultado em 84/4 = 21 pacotes. O
encadeamento aparece no template do segundo passo. No agente de verdade, é o LLM que decide cada
passo à luz das observações (estilo ReAct). Para o projeto, resolva problemas de três etapas.